<a href="https://colab.research.google.com/github/HannahSHeil/ImageProcessingandAnalysis/blob/main/CellSegmentation_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 # 🧪 Nuclei and Cell Segmentation Exercise (Interactive)
 In this notebook, you will:
1. Load a microscopy image
2. Inspect channels (nuclei & cells)
3. Experiment with preprocessing (blur & threshold)
4. Label nuclei
5. Perform cell segmentation with watershed
6. Visualize final overlay
7. Perform measurements and analyse the data

Adapted from © NEUBIAS (https://neubias.github.io/training-resources/workflow_nuclei_and_cells_segmentation/?id_activity_platform=imagej-macro-activity) by H.S.Heil, 04.10.2025

## Step 0 – Install and import libraries

In [1]:
# @title
## Step 1 – Install and import libraries
!pip install scikit-image opencv-python matplotlib numpy tifffile ipywidgets

import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import io, filters, measure, morphology, segmentation, color, exposure
from scipy import ndimage as ndi
import ipywidgets as widgets
from IPython.display import display
from ipywidgets import interactive
import pandas as pd


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 19.0 MB/s eta 0:00:00


## Part 1 - Image Analysis

## Step 1.1 – Load the example image

In [ ]:
# @markdown Run this cell to load the example 2-channel image of adherent cells expressing H2B-mCherry and GFP-tubulin.
# @title
url = "https://github.com/NEUBIAS/training-resources/raw/master/image_data/xyc_16bit__nuclei_tubulin_crop.tif"
img = io.imread(url)
print("Image shape:", img.shape)

# Split into channels
cells = img[0, ...]    # Channel 1 → Tubulin / Cell body
nuclei = img[1, ...]   # Channel 2 → Nuclei


## Step 1.2 – Visualize both channels


In [ ]:
# @markdown Run to visualize both channels
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(cells, cmap='gray');   ax[0].set_title("Cells (Tubulin)")
ax[1].imshow(nuclei, cmap='gray');  ax[1].set_title("Nuclei (DNA stain)")
for a in ax: a.axis('off')
plt.show()


## Step 1.3 – Interactive preprocessing of the nuclei channel

Here we will adjust two parameters and observe the effect:

* Median blur radius → smooths noise.

* Threshold value → separates foreground nuclei from background.

In [ ]:
# @markdown Run to set optimal segmentation parameters
# Convert nuclei image to 8-bit once (for OpenCV compatibility)
nuclei_8bit = cv2.convertScaleAbs(nuclei, alpha=(255.0/np.max(nuclei)))

def preview_nuclei(blur_radius, thresh_val):
    # Median blur (must use 8-bit image)
    nuclei_smooth = cv2.medianBlur(nuclei_8bit, int(blur_radius)*2+1)

    # Convert back to float for thresholding (optional)
    nuclei_binary = nuclei_smooth > (thresh_val / 256.0)  # scaled threshold
    labels = measure.label(nuclei_binary, connectivity=1) # find groups of connected pixels

    fig, ax = plt.subplots(1, 3, figsize=(16,6))
    ax[0].imshow(nuclei_8bit, cmap='gray'); ax[0].set_title("Original (8-bit)")
    ax[1].imshow(nuclei_smooth, cmap='gray'); ax[1].set_title(f"Median blur (radius={blur_radius})")
    ax[2].imshow(color.label2rgb(labels, bg_label=0)); ax[2].set_title(f"Labeled Nuclei (threshold={thresh_val})")
    for a in ax: a.axis('off')
    plt.show()
    print(f"Detected nuclei: {labels.max()}")

blur_slider   = widgets.IntSlider(value=5, min=1, max=15, step=1, description='Blur:')
thresh_slider = widgets.IntSlider(value=63400, min=62000, max=65000, step=100, description='Threshold:')
ui = widgets.HBox([blur_slider, thresh_slider])
out = widgets.interactive_output(preview_nuclei, {'blur_radius': blur_slider, 'thresh_val': thresh_slider})
display(ui, out)


## Step 1.4 – Finalize nuclei segmentation

In [ ]:
# ---------------------------- User Input ----------------------------

# @markdown **Define optimal Nuclei Segmentation Parameters:**

# @markdown Set the following parameters to define how the segmentation should be processed:

blur_radius = 5  #@param {type: "number", min: 1, max: 15, step: 1}
thresh_val = 63400  #@param {type: "number", min: 10000, max: 65535, step: 100}




In [ ]:
# @markdown **Run the nuclear segmentation with optimal parametres:**
# Use chosen parameters (students can adjust below)

nuclei_smooth = cv2.medianBlur(nuclei_8bit, blur_radius*2+1) # blur_radius*2+1 ensures uneven kernel size and that the filter is symmetrical around the central pixel
nuclei_binary = nuclei_smooth > (thresh_val / 256.0) # /256 converts threhold value into 8bit range
nuclei_labels = measure.label(nuclei_binary, connectivity=1)

plt.figure(figsize=(6,6))
plt.imshow(color.label2rgb(nuclei_labels, bg_label=0))
plt.title("Final Nuclei Labels")
plt.axis('off')
plt.show()


## Step 1.5 – Start cell segmentaion: smooth cell signal and prepare cell mask

In [ ]:
# @markdown Run to set optimal segmentation parameters for the tubulin channel

# Convert the cells image to 8-bit once for OpenCV
cells_8bit = cv2.convertScaleAbs(cells, alpha=(255.0/np.max(cells)))

# ---------------------------- Preview Function ----------------------------
def preview_cells(cell_blur, cell_thresh):
    # Smooth cells using mean blur
    cells_smooth = cv2.blur(cells_8bit, (int(cell_blur), int(cell_blur)))

    # Threshold to create binary mask
    cells_binary = cells_smooth > cell_thresh/256


    # Plot results
    fig, ax = plt.subplots(1, 3, figsize=(16,6))
    ax[0].imshow(cells_8bit, cmap='gray'); ax[0].set_title("Original Cells")
    ax[1].imshow(cells_smooth, cmap='gray'); ax[1].set_title(f"Mean Blur (kernel={cell_blur})")
    ax[2].imshow(cells_binary, cmap='gray'); ax[2].set_title(f"Binary mask for Cells (threshold={cell_thresh})")

    for a in ax: a.axis('off')
    plt.show()

    print(f"Detected cells: {cell_labels.max()}")

# ---------------------------- Interactive Widgets ----------------------------
cell_blur_slider = widgets.IntSlider(value=6, min=1, max=15, step=1, description='Blur:')
cell_thresh_slider = widgets.IntSlider(value=63053, min=62000, max=65000, step=1, description='Threshold:')

ui = widgets.HBox([cell_blur_slider, cell_thresh_slider])
out = widgets.interactive_output(preview_cells, {'cell_blur': cell_blur_slider, 'cell_thresh': cell_thresh_slider})

display(ui, out)


In [ ]:
# ---------------------------- User Input ----------------------------

# @markdown **Define optimal Cell Segmentation Parameters:**

# @markdown Set the following parameters to define how the segmentation should be processed:

cell_blur = 6  #@param {type: "number", min: 1, max: 15, step: 1}
cell_thresh = 63053  #@param {type: "number", min: 10000, max: 65535, step: 100}



## Step 7 – Marker-controlled watershed segmentation

In [ ]:
# @markdown ## Run this cell to apply a watershed segmentation to distigush touching cells into single objects
# @markdown To learn more about how the watershed agorithm works visit https://neubias.github.io/training-resources/watershed/

# @title
# Convert the cells image to 8-bit once for OpenCV
cells_8bit = cv2.convertScaleAbs(cells, alpha=(255.0/np.max(cells)))
# Smooth cells using mean blur
cells_smooth = cv2.blur(cells_8bit, (int(cell_blur), int(cell_blur)))

# Threshold to create binary mask
cells_binary = cells_smooth > cell_thresh/256

cells_smooth_invert = np.max(cells_smooth) - cells_smooth
markers = nuclei_labels
mask = cells_binary

cell_labels = segmentation.watershed(cells_smooth_invert, markers=markers, mask=mask)


plt.figure(figsize=(6,6))
plt.imshow(color.label2rgb(cell_labels, bg_label=0))
plt.title("Cell Labels (Watershed Result)")
plt.axis('off')
plt.show()


## Step 8 – Overlay results

In [ ]:
# @title

cells_contrast = exposure.rescale_intensity(cells, in_range='image', out_range=(0,255)).astype(np.uint8)
overlay = color.label2rgb(cell_labels, image=cells_contrast, alpha=0.4, bg_label=0)

plt.figure(figsize=(8,8))
plt.imshow(overlay)
plt.title("Final Segmentation Overlay")
plt.axis('off')
plt.show()

print(f"Total segmented cells: {cell_labels.max()}")


## Part 2: Image Analysis

## Step 2.1 - Take measurements

In [ ]:
# @markdown Run this cell to measure the ...
# @markdown * area,
# @markdown * circularity,
# @markdown * major axis length,
# @markdown * minor axis length,
# @markdown * Postiion in X
# @markdown * Postiion in Y
# @markdown * H2B Signal
# @markdown >... of each segmented nucleus
# Get the region properties for the labeled nuclei
regions = measure.regionprops(nuclei_labels, intensity_image=nuclei)

# Store measurements in a pandas DataFrame
measurements = {
    "Area": [],
    "Circularity": [],
    "Major Axis Length": [],
    "Minor Axis Length": [],
    "Centroid X": [],
    "Centroid Y": [],
    "H2B Signal": [],
}

# Calculate the properties for each region
for region in regions:
    measurements["Area"].append(region.area)
    measurements["Circularity"].append((4 * np.pi * region.area) / (region.perimeter ** 2) if region.perimeter > 0 else 0)  # Circularity formula
    measurements["Major Axis Length"].append(region.major_axis_length)
    measurements["Minor Axis Length"].append(region.minor_axis_length)
    measurements["Centroid X"].append(region.centroid[1])  # Centroid is (y, x), so we use [1] for X
    measurements["Centroid Y"].append(region.centroid[0])  # Centroid is (y, x), so we use [0] for Y
    # Sum of H2B signal inside the nucleus
    measurements["H2B Signal"].append(np.sum(region.intensity_image))

# Create a DataFrame from the measurements dictionary
df_measurements = pd.DataFrame(measurements)

# ---------------------------- Summary Statistics ----------------------------
num_objects = len(df_measurements)
summary_stats = df_measurements.agg(["mean", "std"])

# Display results
print(f"Total number of segmented objects: {num_objects}\n")
print("Mean and standard deviation for each metric:\n")
display(summary_stats)

## Step 2.2 - Explore the data

In [ ]:
# @title
# ---------------------------- Interactive Scatter Plot ----------------------------

# Function to create scatter plot based on selected features
def plot_scatter(x_feature, y_feature):
    plt.figure(figsize=(8, 6))
    plt.scatter(df_measurements[x_feature], df_measurements[y_feature], c='blue', edgecolors='k', alpha=0.7)
    plt.title(f'Scatter Plot: {x_feature} vs. {y_feature}')
    plt.xlabel(x_feature)
    plt.ylabel(y_feature)
    plt.grid(True)
    plt.show()

# Dropdowns for selecting x and y features for scatter plot
x_dropdown = widgets.Dropdown(
    options=df_measurements.columns,
    value="Area",  # default value
    description="X Feature:",
)

y_dropdown = widgets.Dropdown(
    options=df_measurements.columns,
    value="Circularity",  # default value
    description="Y Feature:",
)

# Interactive widget to update the scatter plot based on selected features
interactive_plot = interactive(plot_scatter, x_feature=x_dropdown, y_feature=y_dropdown)
display(interactive_plot)


## Step 2.3 - Your turn!
